## Stage III model plots

Builds figures from **`output/evaluation.csv`**, **`model1_predictions.csv`**, **`model2_predictions.csv`**, and **`output/interpretability/*.csv`**. Saves PNGs under **`output/figures/`**.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "output"
FIG = OUT / "figures"
INTERP = OUT / "interpretability"
FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

RATING_LABELS = [1.0, 2.0, 3.0, 4.0, 5.0]


In [ ]:
def plot_evaluation_metrics() -> None:
    df = pd.read_csv(OUT / "evaluation.csv")
    metrics = ["f1", "accuracy", "weighted_precision", "weighted_recall"]
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(metrics))
    w = 0.35
    m1 = df[df["model"] == "model1"].iloc[0]
    m2 = df[df["model"] == "model2"].iloc[0]
    ax.bar(x - w / 2, [m1[m] for m in metrics], width=w, label="model1 (Random Forest)", color="#2ca25f")
    ax.bar(x + w / 2, [m2[m] for m in metrics], width=w, label="model2 (Naive Bayes)", color="#de2d26")
    ax.set_xticks(x)
    ax.set_xticklabels(["F1", "Accuracy", "Weighted precision", "Weighted recall"], rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
    ax.set_title("Stage III — test-set metrics (evaluation.csv)")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(FIG / "01_evaluation_metrics.png", dpi=150)
    plt.show()
    plt.close(fig)


def plot_confusion(name: str, pred_path: Path, normalize: bool = True) -> None:
    df = pd.read_csv(pred_path)
    y_true = df["label"].values
    y_pred = df["prediction"].values
    labels = RATING_LABELS
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        disp = cm.astype(float) / row_sums
        fmt = ".2f"
        title_suffix = " (row-normalized)"
    else:
        disp = cm
        fmt = "d"
        title_suffix = " (counts)"
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        disp,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=[int(x) for x in labels],
        yticklabels=[int(x) for x in labels],
        vmin=0,
        vmax=1 if normalize else None,
        cbar_kws={"label": "Share of true label" if normalize else "Count"},
        ax=ax,
    )
    ax.set_xlabel("Predicted rating")
    ax.set_ylabel("True rating")
    ax.set_title(f"{name}{title_suffix}")
    fig.tight_layout()
    suffix = "normalized" if normalize else "counts"
    fig.savefig(FIG / f"02_confusion_{name}_{suffix}.png", dpi=150)
    plt.show()
    plt.close(fig)


def plot_rf_importance(top_n: int = 15) -> None:
    path = INTERP / "rf_feature_importance.csv"
    if not path.is_file():
        print("skip RF importance:", path)
        return
    df = pd.read_csv(path).head(top_n)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(df))))
    ax.barh(df["feature_name"][::-1], df["importance"][::-1], color="#2ca25f")
    ax.set_xlabel("Gini importance")
    ax.set_title(f"Random Forest — top {top_n} features (model1)")
    fig.tight_layout()
    fig.savefig(FIG / "03_rf_feature_importance.png", dpi=150)
    plt.show()
    plt.close(fig)


def plot_nb_priors() -> None:
    path = INTERP / "nb_class_priors.csv"
    if not path.is_file():
        print("skip NB priors:", path)
        return
    df = pd.read_csv(path)
    logp = df["log_prior"].values
    priors = np.exp(logp - np.max(logp))
    priors = priors / priors.sum()
    fig, ax = plt.subplots(figsize=(7, 4))
    labs = [str(int(x)) if float(x).is_integer() else str(x) for x in df["label_value"]]
    ax.bar(labs, priors, color="#de2d26")
    ax.set_xlabel("Rating class (label)")
    ax.set_ylabel("Prior probability (normalized from log_prior)")
    ax.set_title("Naive Bayes — class priors (model2)")
    fig.tight_layout()
    fig.savefig(FIG / "04_nb_class_priors.png", dpi=150)
    plt.show()
    plt.close(fig)


def plot_nb_theta_heatmap() -> None:
    path = INTERP / "nb_theta_long.csv"
    if not path.is_file():
        print("skip NB theta:", path)
        return
    df = pd.read_csv(path)
    pivot = df.pivot_table(
        index="feature_name",
        columns="label_value",
        values="log_conditional_probability",
        aggfunc="first",
    )
    fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlBu_r", ax=ax, cbar_kws={"label": "log P(feature|class)"})
    ax.set_title("Naive Bayes — log conditional weights (model2, features_nb)")
    ax.set_xlabel("True rating class")
    fig.tight_layout()
    fig.savefig(FIG / "05_nb_theta_heatmap.png", dpi=150)
    plt.show()
    plt.close(fig)


def write_manifest() -> None:
    manifest = {
        "figures": [
            "01_evaluation_metrics.png — bar comparison from evaluation.csv",
            "02_confusion_model1_rf_normalized.png — RF confusion (row-normalized)",
            "02_confusion_model2_nb_normalized.png — NB confusion (row-normalized)",
            "03_rf_feature_importance.png — RF Gini importances",
            "04_nb_class_priors.png — NB class priors",
            "05_nb_theta_heatmap.png — NB log conditionals (features_nb)",
        ]
    }
    (FIG / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


In [ ]:
plot_evaluation_metrics()


In [ ]:
from IPython.display import Markdown, display

_ev = pd.read_csv(OUT / "evaluation.csv")
_m1 = _ev[_ev["model"] == "model1"].iloc[0]
_m2 = _ev[_ev["model"] == "model2"].iloc[0]
display(
    Markdown(
        "**Numbers from `evaluation.csv` (same plot):**\n\n"
        f"| Metric | model1 (RF) | model2 (NB) |\n"
        f"| --- | --- | --- |\n"
        f"| F1 | {_m1["f1"]:.6f} | {_m2["f1"]:.6f} |\n"
        f"| Accuracy | {_m1["accuracy"]:.6f} | {_m2["accuracy"]:.6f} |\n"
        f"| Weighted precision | {_m1["weighted_precision"]:.6f} | {_m2["weighted_precision"]:.6f} |\n"
        f"| Weighted recall | {_m1["weighted_recall"]:.6f} | {_m2["weighted_recall"]:.6f} |"
    )
)


### Interpretation — test-set metrics

These bars summarise **`evaluation.csv`** on the held-out test partition (weighted metrics except **F1** as emitted by the Spark job).

- **Random Forest (model1)** should exceed **Naive Bayes (model2)** on **F1**, **accuracy**, and weighted precision/recall when the forest captures non-linear structure and class imbalance better than multinomial NB on `features_nb`.
- **Naive Bayes** uses only discrete / scaled sparse inputs (`features_nb`); if its bars sit near zero while RF is healthy, treat NB as a **weak baseline** for this dataset rather than a deployable model.
- Use **F1** first for multi-class imbalance; **accuracy** can mask dominance of rating **5**.

Re-run the previous cell after regenerating **`evaluation.csv`** so numbers stay aligned with the figure.


In [ ]:
plot_confusion("model1_rf", OUT / "model1_predictions.csv", normalize=True)


### Interpretation — Random Forest confusion (row-normalised)

Each **row** is the **true** rating (1–5); **columns** are **predicted** ratings. Values are **fractions of that true class** assigned to each prediction.

- Mass **off the diagonal** means systematic mis-ranking (e.g. predicting **5** when the true label is **1–4**).
- A **strong column** on predicted **5** (or another star) indicates **majority-class collapse**: the model defaults to the modal rating.
- Compare rows for rare stars (**1–2**): if those rows spread across predictions, minority classes remain hard.

Interpret together with **`rf_feature_importance`** (which inputs drive splits).


In [ ]:
plot_confusion("model2_nb", OUT / "model2_predictions.csv", normalize=True)


### Interpretation — Naive Bayes confusion (row-normalised)

Same layout as the RF matrix: **true** on rows, **predicted** on columns, **row-normalised**.

- If **almost all mass sits in one predicted column**, NB has **collapsed** to a single label for most inputs (common when likelihoods differ little across classes on sparse binary/OHE features).
- Poor spread on the diagonal usually aligns with **low F1** in **`evaluation.csv`** for **model2**.
- Row-normalisation highlights **per-class recall patterns**; compare with RF to see whether NB misses specific stars uniformly.


In [ ]:
plot_rf_importance()


### Interpretation — Random Forest feature importance

Bars show **Gini importance** from **`interpretability/rf_feature_importance.csv`** (ranked).

- Higher importance means more tree splits used that coordinate; it is **not** causal effect size.
- **`price`**, **`average_rating`**, **`rating_number`**, and calendar fields often dominate when they correlate with the rating label distribution.
- One-hot **`main_category=…`** entries show **incremental** lift for named categories versus the reference category omitted by `dropLast`.
- **`verified_purchase_num`** at or near **0** suggests verification adds little marginal signal once other fields are present (or is constant in the training slice).

Use this list to prioritise **feature audit** and leakage checks (e.g. target-like fields).


In [ ]:
plot_nb_priors()


### Interpretation — Naive Bayes class priors

Bars convert **`log_prior`** from **`nb_class_priors.csv`** to **normalised probabilities** over rating classes.

- The prior reflects **class frequencies** in training data after the Stage III split (with smoothing in the exported model).
- A dominant prior on **5 stars** matches typical review skew and can reinforce **predicting the majority class** when likelihoods are flat.
- Large gaps between priors explain part of NB decision bias even before conditioning on features.


In [ ]:
plot_nb_theta_heatmap()


### Interpretation — NB log conditional heatmap

Heatmap values are **`log P(feature_j | class)`** from **`nb_theta_long.csv`** for **`features_nb`** only (verified flag + selected one-hot dimensions).

- Compare **across columns** (rating classes) for the **same feature row**: larger differences mean that feature distinguishes stars under the multinomial model.
- Values are on **log scale**; small numeric shifts can correspond to multiplicative probability differences.
- With **few rows**, interpret cautiously: NB uses only the NB feature vector; RF uses the richer **`features`** vector, so this map explains NB behaviour, not the forest.


In [ ]:
write_manifest()
print("Saved PNGs and manifest under", FIG)
